# Visual spot check of EMBED alignment

Samples a few images at random from each (manufacturer, image type, view) group in the EMBED metadata. For each one it shows two panels:

- **Original:** the image as stored. The bars mark where the metadata says the anatomy is: **cyan** for the chest wall edge and **yellow** for the superior edge (lateral, for CC-type views).
- **Realigned:** the image after `Alignment.realign_image`. Every image should show the chest wall on the **left** and superior/lateral at the **top**.

Red boxes are the ROIs from `ROI_coords`, flipped with `realign_box`. They should sit on the same tissue in both panels.

For DBT volumes the middle frame is shown, and ROIs are drawn whatever their frame.

> **Privacy:** the outputs of this notebook contain patient images. Clear all outputs before committing it, for example with `uv run --group validation jupyter nbconvert --clear-output --inplace validation/visual_spot_check.ipynb`, or with *Clear All Outputs* in your editor.

## Configuration

In [ ]:
from pathlib import Path

METADATA_CSV = Path("path/to/metadata_all_cohorts_v1c.csv")

# anon_dicom_path values may point at another mount; rewrite their prefix if needed.
PATH_PREFIX_FROM = ""  # e.g. "/mnt/NAS2/"
PATH_PREFIX_TO = ""    # e.g. "/Volumes/embed/"

IMAGE_TYPES = ["2D", "3D", "cview"]
N_PER_GROUP = 3        # images sampled per (Manufacturer, FinalImageType, ViewPosition) group
MIN_GROUP_SIZE = 50    # skip groups smaller than this
FOCUS = None           # optional pandas query, e.g. "Manufacturer.str.startswith('GE')" or "ViewPosition in ['LM', 'FB']"
THUMBNAIL_SIZE = 384   # longest side of each panel, in pixels
SEED = 0

## Load metadata and sample images

In [ ]:
import ast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from matplotlib.patches import Rectangle
from pydicom.pixels import pixel_array

from mg_aligner import Alignment, Edge

COLUMNS = [
    "anon_dicom_path", "PatientOrientation", "ImageLateralityFinal", "ViewPosition",
    "FieldOfViewRotation", "FieldOfViewHorizontalFlip", "Manufacturer", "ManufacturerModelName",
    "FinalImageType", "ROI_coords",
]
GROUP = ["Manufacturer", "FinalImageType", "ViewPosition"]

meta = pd.read_csv(METADATA_CSV, usecols=COLUMNS, dtype=str, low_memory=False)
meta = meta[meta["FinalImageType"].isin(IMAGE_TYPES) & meta["anon_dicom_path"].notna()]
meta["ViewPosition"] = meta["ViewPosition"].str.strip().str.upper()
if FOCUS:
    meta = meta.query(FOCUS, engine="python")

sizes = meta.groupby(GROUP, dropna=False)["anon_dicom_path"].transform("size")
eligible = meta[sizes >= max(MIN_GROUP_SIZE, N_PER_GROUP)]
sample = eligible.groupby(GROUP, dropna=False).sample(n=N_PER_GROUP, random_state=SEED)
sample["alignment"] = [Alignment.from_metadata(row) for _, row in sample.iterrows()]

print(f"{len(sample)} images sampled from {sample.groupby(GROUP, dropna=False).ngroups} groups")
sample.groupby(GROUP, dropna=False).size().rename("sampled").to_frame()

## Helpers

In [ ]:
EDGE_COLORS = {"chest_wall": "cyan", "superior": "yellow"}


def resolve(path: str) -> Path:
    if PATH_PREFIX_FROM and path.startswith(PATH_PREFIX_FROM):
        path = PATH_PREFIX_TO + path[len(PATH_PREFIX_FROM):]
    return Path(path)


def load_middle_frame(path: Path):
    """Return the header, the middle frame (or the only frame), its index, and the frame count."""
    header = pydicom.dcmread(path, stop_before_pixels=True)
    frames = int(getattr(header, "NumberOfFrames", 1) or 1)
    index = frames // 2
    pixels = pixel_array(path, index=index) if frames > 1 else pixel_array(path)
    return header, pixels, index, frames


def to_display(pixels: np.ndarray, header) -> np.ndarray:
    lo, hi = np.percentile(pixels, [1, 99.5])
    image = np.clip((pixels.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)
    if getattr(header, "PhotometricInterpretation", "") == "MONOCHROME1":
        image = 1 - image
    return image


def parse_boxes(value) -> list:
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        boxes = ast.literal_eval(value)
    except (SyntaxError, ValueError):
        return []
    return [tuple(box) for box in boxes if len(box) == 4]


def mark_edges(ax, alignment: Alignment, height: int, width: int) -> None:
    """Draw bars on the edges where ``alignment`` says the chest wall and superior side are."""
    for field, color in EDGE_COLORS.items():
        edge = getattr(alignment, field)
        if edge is Edge.LEFT:
            ax.axvline(0, color=color, linewidth=5)
        elif edge is Edge.RIGHT:
            ax.axvline(width - 1, color=color, linewidth=5)
        elif edge is Edge.TOP:
            ax.axhline(0, color=color, linewidth=5)
        elif edge is Edge.BOTTOM:
            ax.axhline(height - 1, color=color, linewidth=5)


def show(ax, image: np.ndarray, boxes: list, alignment: Alignment, title: str) -> None:
    """Show ``image`` downsampled with its ROI boxes and the edges ``alignment`` claims."""
    stride = max(1, int(np.ceil(max(image.shape) / THUMBNAIL_SIZE)))
    thumb = image[::stride, ::stride]
    ax.imshow(thumb, cmap="gray", vmin=0, vmax=1)
    for y_min, x_min, y_max, x_max in boxes:
        ax.add_patch(Rectangle(
            (x_min / stride - 0.5, y_min / stride - 0.5),
            (x_max - x_min + 1) / stride, (y_max - y_min + 1) / stride,
            fill=False, edgecolor="red", linewidth=1.5,
        ))
    if alignment.is_known:
        mark_edges(ax, alignment, *thumb.shape)
    ax.set_title(title, fontsize=8)
    ax.axis("off")


def can_realign(alignment: Alignment) -> bool:
    return alignment.is_known and alignment.chest_wall in (Edge.LEFT, Edge.RIGHT)

## Thumbnails

There is one figure per group. Panel titles give the row index in the metadata CSV (to trace an image), the alignment from the CSV metadata, and whether the alignment read from the DICOM header agrees with it. When the CSV gives no alignment (for example multi-frame DBT, whose orientation and laterality live in per-frame functional groups the CSV doesn't flatten), the header's alignment is used and the title says so.

In [ ]:
reference = Alignment.reference()

for group, rows in sample.groupby(GROUP, dropna=False):
    fig, axes = plt.subplots(len(rows), 2, figsize=(7, 3.4 * len(rows)), squeeze=False)
    fig.suptitle(" · ".join(str(value) for value in group), fontsize=11)

    for (index, row), (ax_original, ax_realigned) in zip(rows.iterrows(), axes):
        alignment = row["alignment"]
        path = resolve(row["anon_dicom_path"])
        try:
            header, pixels, frame, frames = load_middle_frame(path)
        except Exception as error:  # missing file, unsupported compression, ...
            for ax in (ax_original, ax_realigned):
                ax.axis("off")
            ax_original.set_title(f"row {index}: could not load ({type(error).__name__})\n{path.name}", fontsize=8)
            continue

        image = to_display(pixels, header)
        boxes = parse_boxes(row["ROI_coords"])
        header_alignment = Alignment.from_metadata(header)
        if not alignment.is_known and header_alignment.is_known:
            alignment, header_note = header_alignment, "CSV unknown · using header"
        elif header_alignment == alignment:
            header_note = "header agrees"
        else:
            header_note = f"header: {header_alignment.chest_wall.value}/{header_alignment.superior.value}"
        claimed = f"{alignment.chest_wall.value}/{alignment.superior.value}"
        frame_note = f" · frame {frame + 1}/{frames}" if frames > 1 else ""

        show(ax_original, image, boxes, alignment, f"row {index} · original · claims {claimed}{frame_note}\n{header_note}")

        if can_realign(alignment):
            height, width = image.shape
            realigned = alignment.realign_image(image)
            realigned_boxes = alignment.realign_boxes(boxes, height=height, width=width)
            show(ax_realigned, realigned, realigned_boxes, reference, "realigned (chest wall left, superior top)")
        else:
            ax_realigned.axis("off")
            ax_realigned.set_title(f"cannot realign: {claimed}", fontsize=8)

    fig.tight_layout()
    plt.show()

## Inspect where one image keeps its orientation

Set `ROW` to a row index from a panel title. The cell prints the top-level orientation attributes, then the ones multi-frame (DBT) objects keep in their functional groups, so you can see why an image's alignment is unknown.

In [ ]:
ROW = None  # e.g. 123456

ATTRIBUTES = ["SOPClassUID", "Manufacturer", "ManufacturerModelName", "PatientOrientation", "ImageLaterality",
              "Laterality", "ViewPosition", "FieldOfViewRotation", "FieldOfViewHorizontalFlip", "NumberOfFrames"]
FRAME_MACROS = {"PatientOrientationInFrameSequence": "PatientOrientation", "FrameAnatomySequence": "FrameLaterality"}

if ROW is not None:
    header = pydicom.dcmread(resolve(meta.loc[ROW, "anon_dicom_path"]), stop_before_pixels=True)
    for name in ATTRIBUTES:
        value = header.get(name)
        if name == "SOPClassUID" and value is not None:
            value = f"{value} ({value.name})"
        print(f"{name:28} {value}")
    for groups in ("SharedFunctionalGroupsSequence", "PerFrameFunctionalGroupsSequence"):
        items = header.get(groups)
        first = items[0] if items else None
        for macro, name in FRAME_MACROS.items():
            inner = first.get(macro) if first is not None else None
            value = inner[0].get(name) if inner else None
            print(f"{groups}[0].{macro}[0].{name}: {value}")
    print("Alignment.from_metadata(header):", Alignment.from_metadata(header))

## What to look for

- **Chest wall (cyan):** it should run along the chest-wall side of the breast, opposite the nipple. On MLO views the pectoral muscle sits there.
- **Superior (yellow):** on MLO and ML/LM views it should run along the top of the breast, where the pectoral muscle and axilla are. On CC-type views it marks the lateral side, which is harder to see. The technologist's radiopaque laterality marker is usually placed laterally (CC) or superiorly (MLO), so it should sit near the yellow bar.
- **Realigned panels:** every one should look the same way round, with the chest wall on the left and the nipple pointing right.
- **ROIs:** the boxes should cover the same tissue before and after realignment.

Write down the row index of any image that looks wrong; the row index plus the metadata CSV is enough to trace it.